# 1. Get and build an area

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khoshkhah/duckOSM/blob/main/docs/notebooks/01_area_to_network.ipynb) <a href="https://khoshkhah.github.io/duckOSM/notebooks/01_area_to_network.ipynb" download><img src="https://img.shields.io/badge/Download-notebook-F37626?logo=jupyter&logoColor=white" alt="Download the notebook"></a>

<img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/banner-1.svg" alt="duckOSM notebook manual, chapter 1: Get and build an area" width="100%">

| **You learn** | the parts of a duckOSM build; seven ways to choose an area; the modes (driving, walking, cycling); your own PBF file; the build options and `config=`; how you save a network |
|---|---|
| **Time** | about 3 minutes (the first download included) |
| **Extras** | `duckosm[widget,viz]` (the install cell installs them) |
| **Next** | [2. Look inside the file](02_inspect.md): the tables, the columns and SQL |

In [1]:
# Install duckOSM when it is not installed (for example in Colab). It takes approximately 1 minute.
# Until duckOSM 0.9.0 is on PyPI, the SDK of these notebooks is in the nightly wheel of main (the GitHub pre-release "nightly").
import importlib.util, json, os, urllib.request
def nightly_wheel():       # the address of the newest nightly wheel
    release = json.load(urllib.request.urlopen("https://api.github.com/repos/Khoshkhah/duckOSM/releases/tags/nightly"))
    return next(a["browser_download_url"] for a in release["assets"] if a["name"].endswith(".whl"))
if importlib.util.find_spec("duckosm") is None:      # not installed: install it with the extras of this notebook
    %pip install -q "duckosm[widget,viz] @ {nightly_wheel()}"
import duckosm             # the SDK
if not hasattr(duckosm, "network"):                    # an old duckOSM (0.8.0 or older), without the SDK
    %pip install -q --force-reinstall --no-deps "duckosm @ {nightly_wheel()}"
    raise SystemExit("duckOSM is updated: restart the session (in Colab: Runtime > Restart session), then run this cell again")
print("duckosm", duckosm.__version__)    # the version

duckosm 0.8.0


## What an area is

An **area** is the part of the world whose roads you want in one file: a city, a district, a park, a box around a
point, or a region. duckOSM describes an area with one polygon, the **boundary**. The build keeps the roads in the
boundary. A road that crosses the boundary stays whole, so the network has no cut roads at its edge.

| You give | The boundary that duckOSM makes |
|---|---|
| a place name (`place`) | the polygon of the first answer of Nominatim for the name |
| an OSM id (`osm_id`) | the polygon of that OSM relation |
| a box (`bbox`) | the rectangle: west, south, east, north |
| a point and a distance (`point`, `dist_m`) | the square `dist_m` metres around the point |
| an H3 cell (`h3_cell`) | the hexagon of the cell |
| a polygon (`boundary`) | your polygon, as it is |
| an OSM file (`pbf`) without a boundary | the area of the whole file |

The picture shows two boundaries for "Monaco". The name finds the country, whose boundary includes a large part of
the sea; the OSM id 2220322 is the city. The gray lines are the roads: some cross the city boundary, and they stay
whole.

<img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/area-boundary.png" alt="Two boundaries for Monaco: the country with its sea, and the city" width="100%">

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-key.svg" width="18" height="18" alt=""> **Key idea**
>
> An area is a polygon. Each way to choose an area (a name, an id, a box, a point, a file) ends as one boundary polygon, and the build keeps the roads in it.

## The parts of a build

A duckOSM build has three steps. Each step has its own call, and `duckosm.network` does all three in one call.

<img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/build-flow.svg" alt="The flow of a build: your area, the boundary, the extract, the cut, the network" width="100%">

| Step | What happens | The call | The result |
|---|---|---|---|
| 1. The boundary | The area becomes a polygon: from a name or an OSM id (Nominatim), a box, a point, an H3 cell or your own GeoJSON | part of `duckosm.get` | `<name>.geojson` |
| 2. The OSM data | duckOSM finds the smallest Geofabrik extract that holds the boundary, downloads it once into a cache, and cuts it to the boundary | `duckosm.get` | `<name>.osm.pbf`, and an `Area` |
| 3. The network | The Rust engine reads the cut file and writes the roads of each mode, the nodes, the graph of turns, the base map and more | `area.build` or `duckosm.build` | `<name>.duckdb`, and a `Network` |

The result is **one DuckDB file**. Each mode is a schema in it (`driving`, `walking`, `cycling`), with the tables
`edges` and `nodes`. Each edge has an `edge_id`, a number that stays the same when you build the area again, so your
own data can join to it (chapter 5).

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-key.svg" width="18" height="18" alt=""> **Key idea**
>
> One area is one file. The networks of all modes, the nodes, the turns and the base map of the area are in the same `.duckdb` file, and `edge_id` connects them to your own data.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-tip.svg" width="18" height="18" alt=""> **Tip**
>
> You can open the file with any DuckDB tool (the `duckdb` program, DBeaver, Python's `duckdb`), not only with duckOSM.

### Four calls: `get`, `build`, `network` and `open`

Each call of the SDK does one step. Learn these four first:

| Call | You give | It does | You get |
|---|---|---|---|
| `duckosm.get(...)` | an area: a name, an OSM id, a box, a point ... | finds the boundary, downloads the OSM data and cuts it to the boundary | an **Area**: the files `monaco.osm.pbf` and `monaco.geojson` |
| `area.build(modes=...)` | an Area | makes the road networks from the OSM file | a **Network**: the file `monaco.duckdb` |
| `duckosm.network(...)` | an area, and the modes | `get` and `build` in one call | a **Network** |
| `duckosm.open("monaco.duckdb")` | a network file that you built before | reads the file: no download and no build | a **Network** |

```text
duckosm.get(...) ──> Area (.osm.pbf + .geojson) ──> area.build(...) ──> Network (.duckdb)
└──────────────────────────── duckosm.network(...) ───────────────────────────┘

duckosm.open("monaco.duckdb") ──> Network                    (a file that you built before)
```

| You want to | Use |
|---|---|
| get the network of an area in one step | `duckosm.network` |
| look at the area or the OSM file before the build, or build the same area again with other options | `duckosm.get`, then `area.build` |
| use a network that you built before | `duckosm.open`. It needs only the file: no OSM id and no modes |

`duckosm.build(pbf=..., boundary=...)` is the same as `area.build`, but it takes the two files directly, not an Area
(section "Your own PBF file").

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-key.svg" width="18" height="18" alt=""> **Key idea**
>
> Build one time, open many times. A build reads the OSM data and takes time. `duckosm.open` reads the finished file at once, so the next chapters of this book open `monaco.duckdb` and build nothing.

## Quick start: one call

`duckosm.network` gets the area and builds it. Here the area is the city of Monaco, OSM relation 2220322. The
argument `modes` chooses the networks: driving, walking and cycling.

In [2]:
net = duckosm.network(osm_id=2220322, modes=["driving", "walking", "cycling"], name="monaco")    # get and build Monaco
net                                                                                 # the card of the network

cut: 34461 nodes, 5685 ways, 339 relations

Output: ~/.cache/duckosm/work/monaco-20261121-000513-9bc9/monaco.duckdb

Network('~/.cache/duckosm/work/monaco-20261121-000513-9bc9/monaco.duckdb', 36.7 MB, driving 2840 edges, walking 11938 edges, cycling 11305 edges)

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The progress lines name each step: the boundary from Nominatim, the extract and the cut, then the build. The card at the end shows the file and, for each mode, the number of edges and nodes and the length in km. The file is in duckOSM's work folder, so the card says **not saved**: section "Save your network" shows how you keep it. A `Network` is also the path of its file.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-warning.svg" width="18" height="18" alt=""> **Warning**
>
> The name "Monaco" finds the *country* of Monaco, whose boundary includes a large part of the sea. That is why this chapter uses the OSM id of the city. When a name gives the wrong area, find its OSM id on [openstreetmap.org](https://www.openstreetmap.org) (search, then the "Relation" number) and use `osm_id=`.

## The summary of each mode

`net.summary()` gives one row for each mode: the edges, the nodes, the length in km and the length of the one-way
roads. `net.modes` lists the modes of the file.

In [3]:
print(net.modes)       # the modes in the file
net.summary()          # one row for each mode

['driving', 'walking', 'cycling']


,mode,edges,nodes,km,oneway_km
0,driving,2840,1768,96.2,35.5
1,walking,11938,4589,260.6,0.0
2,cycling,11305,4650,250.0,35.5


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The walking network is much larger than the driving network: it has the footways, the steps and the paths, and both sides of the streets where OSM draws the sidewalks. A two-way road is two edges, one for each direction.

## The modes

A mode is one network. Each mode keeps the roads that it can use, with its own rules.

| Mode | Keeps | Leaves out |
|---|---|---|
| `driving` | the roads for cars (motorway to residential, service roads, ramps), with one-way rules and turn restrictions | footways, paths, steps, private roads |
| `walking` | footways, paths, steps, pedestrian streets, sidewalks, and the roads where people can walk | motorways and roads where walking is not allowed |
| `cycling` | cycleways, roads for bicycles, and footways where you can push the bicycle | motorways, steps |

The picture shows Monaco: in each panel, all the roads are gray, and the roads of that mode have its color. Walking
has the steps and footways of the old town; driving has the tunnel loop in the west and no footways.

<img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/modes-monaco.png" alt="Monaco: the driving, walking and cycling networks" width="100%">

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-key.svg" width="18" height="18" alt=""> **Key idea**
>
> A road can be in more than one mode. A street with sidewalks is in all three, with the same `edge_id` in each mode. So you join your data once, and it is correct in every mode.

`net.walking` (or `net["walking"]`) is the view of one mode: the same calls, without `mode=`.

In [4]:
net.walking            # the card of one mode

mode,edges,nodes,km,oneway_km
walking,11938,4589,260.6,0.0


## Seven ways to choose an area

`duckosm.get` does steps 1 and 2. It does not build. It gives an `Area`: the pair of files of one area.

| `Area` | File | What it is |
|---|---|---|
| `area.boundary` | `<name>.geojson` | the boundary polygon of the area |
| `area.pbf` | `<name>.osm.pbf` | the OSM data inside the boundary (the cut) |

`area.build()` makes the network from these two files. Use `duckosm.get` to look at an area before the build, or to
build it later with other options. Each way below is one argument of `duckosm.get`.

| Argument | The area | Example |
|---|---|---|
| `place` | the first answer of Nominatim for a name | `duckosm.get("Monaco")` |
| `osm_id` | an OSM relation (a city, a district, a park) | `duckosm.get(osm_id=2220322)` |
| `bbox` | a box: west, south, east, north, in degrees | `duckosm.get(bbox=(7.41, 43.72, 7.44, 43.75))` |
| `point` and `dist_m` | the box `dist_m` metres around a point (lon, lat) | `duckosm.get(point=(7.4246, 43.7384), dist_m=500)` |
| `h3_cell` | one H3 cell | `duckosm.get(h3_cell="883969a40bfffff")` |
| `boundary` | your own polygon: a GeoJSON file, a GeoJSON dict, or a shapely or GeoPandas object | `duckosm.get(boundary="district.geojson")` |
| `pbf` (in `network`) | an OSM file that you have | see "Your own PBF file" below |

The extract stays in the cache, so each next area in Monaco needs no download.

### A box around a point

`point` with `dist_m` is the same as OSMnx's `graph_from_point`: a box of 2 × 500 m around the casino square.

In [5]:
casino = duckosm.get(point=(7.4280, 43.7393), dist_m=500, name="casino", out="data")   # an Area: the file and the boundary
casino                                                                                  # the card of the area

cut: 11894 nodes, 1804 ways, 181 relations

,file,size
OSM file,data/casino.osm.pbf,0.3 MB
boundary,data/casino.geojson,0 kB


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The progress line counts the nodes, ways and relations of the cut. The card names the two files: the cut OSM file and the boundary. The box is a small part of the extract.

### A box in degrees

`bbox` takes west, south, east and north, in this order (the order of GeoJSON and of OSMnx 2).

In [6]:
harbour = duckosm.get(bbox=(7.418, 43.730, 7.430, 43.738), name="harbour", out="data")   # the harbour of Monaco
harbour

cut: 9883 nodes, 1713 ways, 170 relations

,file,size
OSM file,data/harbour.osm.pbf,0.2 MB
boundary,data/harbour.geojson,0 kB


### Your own polygon

`boundary` takes a polygon. Here it is a shapely polygon; a GeoJSON file or a GeoPandas object work the same way.
duckOSM writes it to `<out>/<name>.geojson`.

In [7]:
from shapely.geometry import Polygon                       # any object with __geo_interface__ works
monte_carlo = Polygon([(7.420, 43.737), (7.432, 43.737), (7.432, 43.745), (7.420, 43.745)])
mc = duckosm.get(boundary=monte_carlo, name="monte_carlo", out="data")
mc

cut: 17612 nodes, 2387 ways, 208 relations

,file,size
OSM file,data/monte_carlo.osm.pbf,0.3 MB
boundary,data/monte_carlo.geojson,0 kB


### Look before you get: `dry_run`

`dry_run=True` downloads and writes nothing. It tells you which extract the area needs, its size, and whether it is
in the cache already. Use it before a large area.

In [8]:
plan = duckosm.get(osm_id=2220322, dry_run=True)        # no download, no file
{k: plan["extract"][k] for k in ("name", "km2", "bytes", "cached")}

boundary: Monaco, OSM relation 2220322, 2.39 km², from Nominatim

{'name': 'Monaco', 'km2': 184.34333987387117, 'bytes': 708128, 'cached': True}

## Build an area

`area.build` does step 3 for an `Area`. Its arguments are the same as those of `duckosm.network`: `modes`, `output`,
`fixes` and the build options (below).

In [9]:
casino_net = casino.build(modes=["walking"])      # the walking network of the box around the casino
casino_net.summary()

Output: data/casino.duckdb

,mode,edges,nodes,km,oneway_km
0,walking,3366,1272,87.3,0.0


## Choose the area on a map

`duckosm.AreaPicker` is a map in the notebook. Choose an area with one of the tabs (Search, OSM id, Box, H3 cell,
File); the line under the map shows the extract and its size. Click **Get**, or call `picker.get()`. You can also set
the area from Python with `picker.area`: the widget then shows its tab and its value.

![The AreaPicker widget: the OSM id 2220322 gives the city of Monaco](https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/images/area-picker.png)

In [10]:
picker = duckosm.AreaPicker(out="data")                 # the files go into the folder data/
picker.area = {"mode": "osm_id", "osm_id": 2220322}      # the area when you choose none on the map
picker                                                    # the map

In [11]:
area = picker.result or picker.get()    # the result of Get, or get it now
area

boundary: Monaco, OSM relation 2220322, 2.39 km², from Nominatim

,file,size
OSM file,data/monaco.osm.pbf,0.6 MB
boundary,data/monaco.geojson,16 kB


## Your own PBF file

`pbf=` takes an `.osm.pbf` file that you already have: from Geofabrik, from BBBike, from your own OSM editor, or a
file that you cut with osmium. Without `boundary=`, the whole file is the area. With it, duckOSM cuts the file to
the boundary before the build. Here the file is a Monaco extract (0.6 MB).

In [12]:
import urllib.request                                                           # your own file: here, a download
mine = "my_monaco.osm.pbf"
urllib.request.urlretrieve("https://github.com/Khoshkhah/duckOSM/raw/main/tests/data/monaco.osm.pbf", mine)
whole = duckosm.network(pbf=mine, name="monaco_whole", modes=["driving"])        # the whole file
whole.summary()

Output: ~/.cache/duckosm/work/my_monaco-20261121-000526-3fbf/my_monaco.duckdb

,mode,edges,nodes,km,oneway_km
0,driving,2844,1769,96.2,35.6


In [13]:
cut = duckosm.network(pbf=mine, boundary="data/monte_carlo.geojson", name="monte_carlo_cut", modes=["driving"])
cut.summary()                                                                     # only the roads in Monte Carlo

Output: ~/.cache/duckosm/work/my_monaco-20261121-000530-33eb/my_monaco.duckdb

,mode,edges,nodes,km,oneway_km
0,driving,779,484,28.4,8.9


> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The cut to the Monte Carlo polygon (from "Your own polygon" above) keeps about a quarter of the driving edges of the whole file (779 of 2844).

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-tip.svg" width="18" height="18" alt=""> **Tip**
>
> If the download of an extract fails on your machine (some cloud networks block Geofabrik), get the file another way and use `pbf=`.

## The build options

These keywords work in `duckosm.network`, `area.build` and `duckosm.build`. Each one sets one key of the config file.

| Keyword | Default | What it does |
|---|---|---|
| `speeds` | `True` | `False`: no `maxspeed_kmh` (the raw OSM `maxspeed` text stays), and no `cost_s` |
| `costs` | `True` | `False`: no `cost_s`, the travel time of each edge |
| `restrictions` | `True` | `False`: no turn restrictions (`driving.turn_restrictions`) |
| `bus_routes` | `True` | `False`: no bus routes (`bus.routes`, `bus.route_edges`) |
| `connect_paths` | `True` | `False`: the free ends of paths stay free; `True` connects each one to the nearest node |
| `snap_m` | `10` | the largest distance, in metres, for `connect_paths` |
| `features` | `True` | `False`: no base-map layers (`features.*`: water, land, buildings) |
| `levels` | `True` | `False`: no drawing order of the roads (bridges over tunnels) |
| `dem` | none | a DEM file (GeoTIFF): the heights of the nodes and of the edges, after the build |

A smaller build is faster. Here the build has no bus routes, no base map and no drawing order.

In [14]:
small = duckosm.network(osm_id=2220322, name="monaco_small", modes=["driving"],
                        bus_routes=False, features=False, levels=False)       # three options off
sorted({t.split(".")[0] for t in small.tables()})                           # the schemas of the file

cut: 34461 nodes, 5685 ways, 339 relations

Output: ~/.cache/duckosm/work/monaco_small-20261121-000533-7644/monaco_small.duckdb

['driving', 'main']

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The file has only `driving` and `main`: no `bus` (no bus routes), no `features` and `raw` (no base map) and no `visualization` (no drawing order). Compare it with the schemas of `net` from the quick start.

In [15]:
sorted({t.split(".")[0] for t in net.tables()})          # the schemas of a build with the defaults

['bus',
 'cycling',
 'driving',
 'features',
 'main',
 'raw',
 'visualization',
 'walking']

### Every other option: `config=`

The config file has many more keys (`config/template.yaml` in the repository lists them all). `config=` takes such a
file (YAML or JSON) or a Python dict. A keyword wins over the same key of the config.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-warning.svg" width="18" height="18" alt=""> **Warning**
>
> A wrong keyword stops the call with the nearest name, for example `network has no argument 'bus_route': did you mean 'bus_routes'?`.

In [16]:
cfg = {"options": {"bus_routes": False}, "clip": {"connect_snap_m": 5}}      # two keys of the config file
tuned = duckosm.network(osm_id=2220322, name="monaco_tuned", modes=["walking"], config=cfg,
                        features=False, levels=False)
tuned.summary()

cut: 34461 nodes, 5685 ways, 339 relations

Output: ~/.cache/duckosm/work/monaco_tuned-20261121-000534-0f26/monaco_tuned.duckdb

,mode,edges,nodes,km,oneway_km
0,walking,11594,4587,258.0,0.0


In [17]:
try:
    duckosm.network(osm_id=2220322, name="x", modes=["driving"], bus_route=False)    # a wrong name
except duckosm.DuckOSMError as e:
    print(e)

network has no argument 'bus_route': did you mean 'bus_routes'? The area: osm_id, bbox, h3_cell, boundary, point, dist_m, cache, refresh. The build options: speeds, costs, restrictions, bus_routes, connect_paths, snap_m, config, dem, features, levels.


## Save your network

A build writes its files to duckOSM's **work folder**, not to your folder, so your folder stays clean while you try
things. The card of a network says **not saved** while the file is there. `net.save(path)` copies the network to the
place that you choose, with its OSM file and its boundary beside it, and gives the `Network` of the copy.

In [18]:
kept = net.save("monaco.duckdb")
kept

Network('monaco.duckdb', 36.7 MB, driving 2840 edges, walking 11938 edges, cycling 11305 edges)

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-output.svg" width="18" height="18" alt=""> **What the output shows**
>
> The card of the copy has no "not saved" note: `monaco.duckdb` is in your folder. `monaco.osm.pbf` and `monaco.geojson` are beside it, so `kept.rebuild(...)` and `kept.edit()` still work. `kept.saved` is `True`.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-key.svg" width="18" height="18" alt=""> **Key idea**
>
> `monaco.duckdb` is the network of this book. Each next chapter opens it with `duckosm.open("monaco.duckdb")` and builds nothing. To read a saved network, you need only its file: no OSM id, no modes.

> <img src="https://raw.githubusercontent.com/Khoshkhah/duckOSM/main/docs/notebooks/images/icon-warning.svg" width="18" height="18" alt=""> **Warning**
>
> A network that you do not save stays in the work folder only until you clean it with `duckosm.clear_work`. Save each network that you want to keep.

### Write the network to a file at once: `path=`

When you know the file name before the build, give it as `path`. The network then goes there directly, not to the
work folder.

In [19]:
direct = duckosm.network(osm_id=2220322, modes=["driving"], path="monaco_driving.duckdb",
                         features=False, levels=False)
direct.saved

cut: 34461 nodes, 5685 ways, 339 relations

Output: ./monaco_driving.duckdb

True

### Keep the OSM data of an area: `area.save`

`area.save(folder)` copies the two files of an `Area` (the cut OSM file and the boundary) into a folder of yours.

In [20]:
casino.save("casino_area")

,file,size
OSM file,casino_area/casino.osm.pbf,0.3 MB
boundary,casino_area/casino.geojson,0 kB


### Open a saved file again: `duckosm.open`

The saved file stays on disk. `duckosm.open` gives its `Network` in a new session, without a build.

In [21]:
again = duckosm.open("monaco.duckdb")
again.summary()

,mode,edges,nodes,km,oneway_km
0,driving,2840,1768,96.2,35.5
1,walking,11938,4589,260.6,0.0
2,cycling,11305,4650,250.0,35.5


### The work folder

| Files | Where | Set it with |
|---|---|---|
| networks and areas that you did not save | the work folder, `~/.cache/duckosm/work` (`duckosm.work_dir()` gives it) | `DUCKOSM_CACHE` (the work folder is `<cache>/work`) |
| your saved networks | where you save them | `net.save(path)`, `path=`, `area.save(folder)` |
| the Geofabrik extracts | the cache: `~/.cache/duckosm` | `DUCKOSM_CACHE` |

`duckosm.clear_work()` deletes the work folder; `duckosm.clear_work(days=7)` deletes only the parts older than seven
days. Your saved files are not in it, so they stay.

In [22]:
duckosm.clear_work(days=7)

0

## Try it yourself

**1.** Get the walking network of 1 km around a place that you know, with `point=` and `dist_m=1000`.

<details><summary>Show the answer</summary>

```python
mine = duckosm.network(point=(2.2945, 48.8584), dist_m=1000, modes=["walking"], name="eiffel")   # the Eiffel Tower
mine.summary()
```
</details>

**2.** Build Monaco twice, once with `speeds=False`, and find the columns that the second file does not have.

<details><summary>Show the answer</summary>

```python
fast = duckosm.network(osm_id=2220322, name="monaco_nospeed", modes=["driving"], speeds=False)
set(net.columns("driving.edges").column_name) - set(fast.columns("driving.edges").column_name)   # maxspeed_kmh, cost_s
```
</details>

**3.** Use `dry_run=True` for your own city, and look at the size of its extract before you get it.

<details><summary>Show the answer</summary>

```python
plan = duckosm.get("Tartu", dry_run=True)     # any city name
plan["extract"]["name"], round(plan["extract"]["bytes"] / 1e6, 1), "MB"
```
</details>

## Words in this chapter

| Word | Meaning |
|---|---|
| area | the part of the world that you build: a city, a district, a box |
| boundary | the polygon of the area (`<name>.geojson`) |
| extract | a file of OSM data for a country or a region, from Geofabrik |
| PBF | the file format of OSM data (`.osm.pbf`) |
| cut | the part of an extract inside the boundary |
| mode | one network: driving, walking or cycling |
| edge | one direction of a road between two nodes: a row of `<mode>.edges` |
| node | a point where edges meet: a row of `<mode>.nodes` |
| `edge_id` | the number of an edge; it stays the same when you build the area again |

## Summary

| Call | Does |
|---|---|
| `duckosm.network(...)` | gets and builds an area in one call; gives a `Network` |
| `duckosm.get(...)` | the boundary and the cut OSM file; gives an `Area` (`dry_run=True`: only a plan) |
| `area.build(modes=...)` | builds an `Area` |
| `duckosm.build(pbf=..., boundary=...)` | builds from files |
| `duckosm.AreaPicker()` | chooses an area on a map |
| `net.save(path)`, `area.save(folder)` | keeps a network or an area where you choose |
| `duckosm.network(..., path=...)` | builds straight into a file of yours |
| `duckosm.open(path)` | opens a file that you saved before |
| `duckosm.work_dir()`, `duckosm.clear_work()` | the work folder, and its cleaning |
| `net.summary()`, `net.modes`, `net.walking` | the modes of a file |

The reference: [Python SDK, Core](../reference/python.md). **Next:** [2. Look inside the file](02_inspect.md), the
tables, the columns and SQL.